# Control lineage and pruned traces on the weather benchmark

This notebook runs the weather benchmark from `benchmarks/disc` at a small scale. A UDF converts snowfall readings to millimetres, but it treats any unit other than `mm` as feet. One reading recorded in inches therefore becomes about 27,000 mm, and that single reading becomes the maximum for its month.

The bug involves two fields in two different roles. The reading's `value` flows into the result, and its `unit` only decides which branch runs. Prism reports the first as data lineage and the second as control lineage.

In [1]:
import os
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from prism import Prism, udf

# one executor per worker, sized to the worker
spark = (SparkSession.builder.appName("prism-weather")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.executor.cores", os.environ.get("WORKER_CORES", "2"))
         .config("spark.executor.memory", os.environ.get("WORKER_MEMORY", "2g"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
DATA = os.environ.get("PRISM_DATA", "/data")
SCALE = float(os.environ.get("PRISM_SCALE", "1"))   # multiplies every generated table's rows
print("Spark", spark.version, "on", spark.sparkContext.master, "| data scale", SCALE)

Spark 4.1.2 on spark://spark-master:7077 | data scale 1.0


## 1. Generate the benchmark data

The generator is the same one the full-scale benchmarks use. At the default scale it writes 500,000 readings for months 1 to 11. It adds a faulty group of 300 readings for month 12, one of which is recorded in inches, and that group stays at 300 readings at every scale.

In [2]:
import gen   # the benchmark data generator, used here only to write the table

os.environ["GEN_PARTS"] = "16"
gen.gen_one(spark, "weather", n=int(500_000 * SCALE), out=f"{DATA}/disc")
WEATHER = f"{DATA}/disc/weather.parquet"
spark.read.parquet(WEATHER).groupBy("unit").count().show()

wrote ~500000 weather rows (FG=300 faulty group) -> /data/disc/weather.parquet


+----+------+
|unit| count|
+----+------+
|  mm|499999|
|  in|     1|
+----+------+



## 2. Capture and trace the anomalous maximum

The job converts every reading to millimetres with a UDF and takes the maximum per month. Prism is switched on before the job runs, and the job is wrapped in a function because section 5 runs it again.

In [3]:
@udf(DoubleType())
def to_mm(value, unit):
    # bug: any unit other than "mm" is treated as feet, so an inch reading is scaled by 304.8
    return float(value if unit == "mm" else value * 304.8)

def weather_job(p):
    p.enable_capture()                            # record which readings feed each output
    p.enable_udf_aware(to_mm, mode="lazy-cell")   # record which fields to_mm read, per record
    p.enable_influence()                          # record each reading's contribution to the max
    readings = spark.read.parquet(WEATHER)
    result = (readings.withColumn("mm", to_mm("value", "unit"))     # convert every reading
              .groupBy("month").agg(F.max("mm").alias("max_mm")))   # maximum snowfall per month
    return readings, result

p = Prism(spark)
source, result = weather_job(p)
rows = p.collect_with_lineage(result)                    # run the job once, capturing lineage
worst_row, worst_id = max(rows, key=lambda r: r[0]["max_mm"])   # the implausibly large maximum
bad = [worst_id]
print("suspicious output:", worst_row)

cur = p.trace(result, bad)                               # walk back from that output
while not cur.at_scan:
    cur = cur.go_back()
witnesses = cur.show()
print(len(witnesses), "readings traced")

suspicious output: Row(month=12, max_mm=27432.0)


300 readings traced


## 3. Data and control lineage on the faulty reading

The two-phase mode computes each traced record's exact field split at trace time. The faulty reading took the non-`mm` branch, so its value is data and its unit is control.

In [4]:
faulty = [w for w in witnesses if w["unit"] == "in"]
print(faulty[0])
assert faulty[0]["_data"] == ["value"] and faulty[0]["_control"] == ["unit"]

{'value': 90.0, 'unit': 'in', 'month': 12, '_data': ['value'], '_control': ['unit'], '_regime': 'model'}


## 4. Influence on a maximum

For a maximum, only the largest record changes the result when it is removed, and its influence is its margin over the runner-up. Every other record has influence zero. The inch reading is therefore the only record with nonzero influence.

In [5]:
influence = p.trace(result, bad).influence("max_mm")
nonzero = [(i, v) for i, v in influence.scores.items() if v != 0]
print("records with nonzero influence:", nonzero)
assert len(nonzero) == 1

records with nonzero influence: [(12884933344, 27133.0)]


## 5. The pruned source re-read

The last step of a trace turns record identifiers back into source rows by re-reading the table. Each identifier records the file split and the row offset it came from, so Prism reads only the splits that hold the traced records. The setting `spark.prism.trace.prunePartitions` turns this off, which lets us count the tasks each version launches. This small table would normally be packed into a few splits, so the first lines read it again with one split per file, as Spark does for a large table.

In [6]:
spark.conf.set("spark.sql.files.maxPartitionBytes", "256k")
spark.conf.set("spark.sql.files.openCostInBytes", "0")
p = Prism(spark)
source, result = weather_job(p)                          # the same job, read with one split per file
rows = p.collect_with_lineage(result)
bad = [max(rows, key=lambda r: r[0]["max_mm"])[1]]
print("source splits:", source.rdd.getNumPartitions())

def reread_tasks(prune):
    group = f"reread-{prune}"
    spark.conf.set("spark.prism.trace.prunePartitions", str(prune).lower())
    c = p.trace(result, bad)
    while not c.at_scan:
        c = c.go_back()
    sc = spark.sparkContext
    sc.setJobGroup(group, "source re-read")
    t = time.time()
    rows = c.show()
    elapsed = time.time() - t
    sc.setJobGroup("other", "")
    tracker = sc.statusTracker()
    tasks = sum(tracker.getStageInfo(s).numTasks
                for j in tracker.getJobIdsForGroup(group)
                for s in tracker.getJobInfo(j).stageIds)
    return len(rows), tasks, elapsed

full = reread_tasks(False)
pruned = reread_tasks(True)
print(f"full re-read:   {full[0]} rows, {full[1]} tasks, {full[2]:.2f} s")
print(f"pruned re-read: {pruned[0]} rows, {pruned[1]} tasks, {pruned[2]:.2f} s")
assert full[0] == pruned[0] and pruned[1] < full[1]

source splits: 24


full re-read:   300 rows, 28 tasks, 0.30 s
pruned re-read: 300 rows, 5 tasks, 0.15 s


In [7]:
spark.stop()